# AgentCore Registry Demo — Tool Catalog for Open Insurance

This notebook demonstrates the **AWS AgentCore Registry** — a managed discovery and catalog service for MCP servers. While the Gateway **runs** your tools, the Registry **catalogs** them so agents can discover approved tools across your organization.

## Why does the Registry matter?

Without a Registry, agents have **hardcoded tool lists** — every team manually configures which MCP servers their agents can see. This breaks down at scale:

- **Team A** builds a PolicyLookup tool but **Team B** doesn't know it exists, so they build their own
- A new tool is deployed but **agents can't find it** without someone updating their config
- There's **no governance** — anyone can publish anything, with no review or approval process
- When a tool is deprecated, agents still try to call it because **nobody updated the hardcoded list**

The Registry solves this with a **Publish → Discover → Govern** workflow:

```
Teams publish MCP server records     Agents search with natural language
        │                                       │
        ▼                                       ▼
   ┌──────────┐                          "Find tools for
   │ Registry │  ◀── semantic search ──  insurance claims"
   │          │                                 │
   │ PolicyLookup (v1.0, APPROVED)              │
   │ ClaimsData  (v1.0, APPROVED)               ▼
   │ CRMTools    (v2.1, DEPRECATED)     Returns matching records
   │ NewTool     (v0.1, DRAFT)          with tool schemas
   └──────────┘
        │
        │  Governance: DRAFT → review → APPROVED
        │  Deprecated tools hidden from search
        │  EventBridge integration for approval workflows
```

**Key point:** The Registry is a **catalog**, not a runtime. It stores tool metadata (names, descriptions, input schemas) — it does NOT invoke tools. Agents discover tools via the Registry, then invoke them through the **Gateway** (which handles authentication, Cedar policies, and Lambda execution).

| Layer | Role | Analogy |
|-------|------|---------|
| **Registry** | Find the right tool | App store / internal wiki |
| **Gateway** | Run the tool securely | API gateway / runtime |

## Architecture

```
Claude Code / Strands Agent
  │
  ├── MCP ──▶ Registry (discover tools — metadata only)
  │           │  search_registry_records tool
  │           │  JWT auth (Okta PKCE)
  │           │
  │           └── Returns: tool names, schemas, descriptions
  │
  ├── MCP ──▶ Gateway (invoke tools — actual execution)
  │           │  PolicyLookup___lookup_policy
  │           │  ClaimsData___query_claims
  │           │  JWT auth (Okta PKCE) + Cedar ENFORCE
  │           │
  │           ├───────────────┐
  │           ▼               ▼
  │         Lambda          Lambda
  │       (PolicyLookup)  (ClaimsData)
  │
  ▼
Okta (OIDC identity provider)
```

## What this notebook does

1. Install dependencies and load existing Gateway config
2. Create a **Registry** with Okta JWT authorization (same IdP as Gateway)
3. Publish **MCP server records** for PolicyLookup and ClaimsData tools
4. Demo **governance workflow** — submit for approval, approve records
5. **Search** the registry using Okta JWT tokens (PKCE browser flow)
6. Configure **Claude Code** for Registry-first discovery and provide test instructions
7. Publish an **AGENT_SKILLS** record — a claim triage workflow skill
8. Approve and search for skills in the Registry
9. Demo **Discover → Enable → Use** — find a skill, install as slash command, invoke it
10. Document the **inline-only limitation** of AGENT_SKILLS (no code artifacts)
11. Save config, restore Gateway, and clean up

## Prerequisites

- **`00_okta_setup.ipynb`** completed (auth server, SPA app with `OKTA_SPA_CLIENT_ID` in `.env`)
- **`01_agentcore_setup.ipynb`** completed (Gateway, Lambda targets, Cedar policies deployed)
- **`gateway_config.json`** exists with deployed resource IDs
- **`.env`** with `OKTA_SPA_CLIENT_ID` and `OKTA_API_TOKEN`
- **boto3 >= 1.42.89** (upgraded in Cell 1)

## Cell 1: Dependencies + Configuration

Upgrades boto3 for Registry API support and loads the existing Gateway config from `01_agentcore_setup.ipynb` and the SPA client ID from `.env`.

In [ ]:
%pip install -q --upgrade boto3 requests python-dotenv PyJWT

import os
import json
import time
import boto3
import requests
from dotenv import load_dotenv

load_dotenv(override=True)

# --- Load Gateway config from 01_setup ---
with open("gateway_config.json") as f:
    config = json.load(f)

GATEWAY_ID = config["gateway_id"]
GATEWAY_URL = config["gateway_url"]
OKTA_DOMAIN = config["okta_domain"]
OKTA_ISSUER = config["okta_issuer"]
AWS_REGION = config["aws_region"]

# --- SPA client (public, PKCE) — created by 00_okta_setup Cell 4 ---
SPA_CLIENT_ID = os.environ["OKTA_SPA_CLIENT_ID"]
CALLBACK_PORT = config.get("claude_code_callback_port", 8400)

# --- Okta API token (optional — used to register PKCE redirect URI) ---
OKTA_API_TOKEN = os.environ.get("OKTA_API_TOKEN", "")

# --- boto3 clients ---
agentcore_control = boto3.client("bedrock-agentcore-control", region_name=AWS_REGION)
agentcore_data = boto3.client("bedrock-agentcore", region_name=AWS_REGION)

# --- Verify Registry API is available ---
assert hasattr(agentcore_control, "create_registry"), \
    f"boto3 {boto3.__version__} lacks Registry API — need >= 1.42.89"

print(f"boto3 version:   {boto3.__version__}")
print(f"Gateway ID:      {GATEWAY_ID}")
print(f"Okta Domain:     {OKTA_DOMAIN}")
print(f"Okta Issuer:     {OKTA_ISSUER}")
print(f"SPA Client ID:   {SPA_CLIENT_ID}")
print(f"Callback Port:   {CALLBACK_PORT}")
print(f"AWS Region:      {AWS_REGION}")
print(f"\n✅ Configuration loaded")

## Cell 2: Create the Registry

Creates an AgentCore Registry with **Okta JWT authorization** — the same identity provider used by the Gateway. This means agents authenticate once with Okta and can both discover tools (Registry) and invoke them (Gateway).

Key settings:
- **`authorizerType: CUSTOM_JWT`** — validates Okta JWTs against the OIDC discovery endpoint
- **`allowedAudience: ["api://default"]`** — Okta's default authorization server audience
- **`allowedClients`** — accepts tokens from the SPA client (Authorization Code + PKCE)
- **`autoApproval: False`** — requires explicit approval before records are searchable (governance demo)

In [ ]:
REGISTRY_NAME = "open-insurance-registry"
OKTA_DISCOVERY_URL = f"{OKTA_ISSUER}/.well-known/openid-configuration"

allowed_clients = [SPA_CLIENT_ID]

print(f"Registry name:    {REGISTRY_NAME}")
print(f"Discovery URL:    {OKTA_DISCOVERY_URL}")
print(f"Allowed clients:  {allowed_clients}")

# --- Create the Registry ---
print(f"\nCreating Registry with Okta JWT auth...")

try:
    resp = agentcore_control.create_registry(
        name=REGISTRY_NAME,
        description="Open Insurance tool catalog — discover MCP servers for policy and claims operations",
        authorizerType="CUSTOM_JWT",
        authorizerConfiguration={
            "customJWTAuthorizer": {
                "discoveryUrl": OKTA_DISCOVERY_URL,
                "allowedAudience": ["api://default"],
                "allowedClients": allowed_clients,
            }
        },
        approvalConfiguration={"autoApproval": False},
    )
    REGISTRY_ARN = resp["registryArn"]
    # Extract registry ID from ARN (format: arn:aws:bedrock-agentcore:region:account:registry/registryId)
    REGISTRY_ID = REGISTRY_ARN.rsplit("/", 1)[-1]
    print(f"Created Registry: {REGISTRY_ID}")
except agentcore_control.exceptions.ConflictException:
    # Registry already exists — find it
    print("Registry already exists — looking up...")
    registries = agentcore_control.list_registries()
    for reg in registries.get("registries", registries.get("items", [])):
        if reg["name"] == REGISTRY_NAME:
            REGISTRY_ID = reg["registryId"]
            REGISTRY_ARN = reg["registryArn"]
            print(f"Found existing Registry: {REGISTRY_ID}")
            break
    else:
        raise RuntimeError(f"ConflictException but could not find registry '{REGISTRY_NAME}'")

# --- Poll for READY status ---
print("Waiting for Registry to be READY...")
for attempt in range(30):
    reg = agentcore_control.get_registry(registryId=REGISTRY_ID)
    status = reg["status"]
    if status == "READY":
        REGISTRY_ARN = reg["registryArn"]
        print(f"\n✅ Registry READY")
        print(f"  Registry ID:  {REGISTRY_ID}")
        print(f"  Registry ARN: {REGISTRY_ARN}")
        break
    elif status in ("FAILED", "CREATE_FAILED"):
        print(f"Registry FAILED: {reg}")
        break
    time.sleep(5)
    print(f"  Status: {status} ({(attempt + 1) * 5}s)")
else:
    print("Warning: Registry did not become READY in time")

## Cell 3: Publish MCP Server Records

Each Registry record represents an MCP server with its tool schemas. We register the same tools that are deployed as Gateway targets:

| Record | Tool | Description | Gateway Target |
|--------|------|-------------|----------------|
| `open-insurance-policy-lookup` | `lookup_policy` | Policy details by number | PolicyLookup Lambda |
| `open-insurance-claims-data` | `query_claims` | Confidential claims query | ClaimsData Lambda |

Records start in **DRAFT** status — they are not searchable until approved (Cell 4).

### MCP Descriptor Format

Each record carries two descriptors:
- **`server`** — MCP server metadata (name, description, version)
- **`tools`** — Tool definitions with input schemas (same format as Gateway target `inlinePayload`)

### Claude Code Installation Instructions in Descriptions

Record descriptions now include **structured installation instructions** with the full `claude mcp add-json` command and MCP server config JSON. This enables a natural "discover → install" flow: when Claude Code searches the Registry and finds a tool, it can read the description and directly install the Gateway as an MCP server — no manual configuration needed.

After installation, `/mcp` or `/reload-plugins` in Claude Code connects to the new server without restarting the session.

In [ ]:
# --- Tool schemas (same as Gateway targets in 01_setup) ---

POLICY_LOOKUP_TOOL = {
    "name": "lookup_policy",
    "description": "Look up insurance policy details by policy number. Available policies: POL-10042, POL-10078, POL-10103, POL-10156, POL-10201.",
    "inputSchema": {
        "type": "object",
        "properties": {
            "policy_number": {"type": "string", "description": "Policy number (e.g., POL-10042)"}
        },
        "required": ["policy_number"]
    }
}

CLAIMS_TOOL = {
    "name": "query_claims",
    "description": "Query insurance claims data. Contains CONFIDENTIAL claims information including adjuster notes. Search by claim ID, policy number, holder name, or ask for 'claims summary'. Use max_amount to filter by claim size.",
    "inputSchema": {
        "type": "object",
        "properties": {
            "query": {"type": "string", "description": "Claims query — e.g., 'claims summary', 'CLM-2024-0891', 'POL-10042', or a holder name"},
            "max_amount": {"type": "integer", "description": "Maximum claim amount to return (e.g., 100000 for claims <= $100K)"}
        },
        "required": ["query"]
    }
}

# --- Claude Code MCP server config (embedded in record descriptions) ---

GATEWAY_MCP_CONFIG = {
    "type": "http",
    "url": GATEWAY_URL,
    "oauth": {
        "clientId": SPA_CLIENT_ID,
        "callbackPort": CALLBACK_PORT,
        "scope": "openid groups",
        "authorizationUrl": f"{OKTA_ISSUER}/v1/authorize",
        "tokenUrl": f"{OKTA_ISSUER}/v1/token",
    },
}
GATEWAY_MCP_CONFIG_JSON = json.dumps(GATEWAY_MCP_CONFIG)

# --- Define registry records ---

records_to_create = [
    {
        "name": "open-insurance-policy-lookup",
        "description": (
            "Insurance policy lookup MCP server — returns policy details by policy number "
            "(POL-10042, POL-10078, POL-10103, POL-10156, POL-10201). "
            "Available to all authenticated users via Cedar ENFORCE mode.\n\n"
            "## Claude Code Installation\n\n"
            "To add the AgentCore Gateway (which hosts this tool) as an MCP server in Claude Code, run:\n\n"
            "```\n"
            f"claude mcp add-json agentcore-gateway '{GATEWAY_MCP_CONFIG_JSON}'\n"
            "```\n\n"
            "Then run `/mcp` or `/reload-plugins` in Claude Code to connect without restarting the session. "
            "An Okta login will open in your browser for authentication.\n\n"
            "Gateway tool name: `PolicyLookup___lookup_policy`\n"
            f"Gateway URL: {GATEWAY_URL}\n"
            "Auth: Okta JWT (CUSTOM_JWT authorizer), Cedar ENFORCE mode"
        ),
        "server_name": "open-insurance/policy-lookup",
        "server_description": "Insurance policy lookup MCP server for the Open Insurance platform",
        "tools": [POLICY_LOOKUP_TOOL],
    },
    {
        "name": "open-insurance-claims-data",
        "description": (
            "Confidential insurance claims data MCP server — query claims by ID, policy number, "
            "holder name, or ask for a claims summary. Includes adjuster notes, fraud flags, "
            "and claim amounts up to $250K. "
            "Access restricted by Cedar policy — only claims adjusters (e.g., Bob) have full access.\n\n"
            "## Claude Code Installation\n\n"
            "To add the AgentCore Gateway (which hosts this tool) as an MCP server in Claude Code, run:\n\n"
            "```\n"
            f"claude mcp add-json agentcore-gateway '{GATEWAY_MCP_CONFIG_JSON}'\n"
            "```\n\n"
            "Then run `/mcp` or `/reload-plugins` in Claude Code to connect without restarting the session. "
            "An Okta login will open in your browser for authentication. "
            "Note: Cedar policies control access — you must be an authorized claims adjuster to use this tool.\n\n"
            "Gateway tool name: `ClaimsData___query_claims`\n"
            f"Gateway URL: {GATEWAY_URL}\n"
            "Auth: Okta JWT (CUSTOM_JWT authorizer), Cedar ENFORCE mode"
        ),
        "server_name": "open-insurance/claims-data",
        "server_description": "Insurance claims data MCP server with confidential adjuster notes",
        "tools": [CLAIMS_TOOL],
    },
]

# --- Create records ---
record_ids = {}

for rec in records_to_create:
    try:
        resp = agentcore_control.create_registry_record(
            registryId=REGISTRY_ID,
            name=rec["name"],
            description=rec["description"],
            descriptorType="MCP",
            recordVersion="1.0.0",
            descriptors={
                "mcp": {
                    "server": {
                        "schemaVersion": "2025-12-11",
                        "inlineContent": json.dumps({
                            "name": rec["server_name"],
                            "description": rec["server_description"],
                            "version": "1.0.0"
                        })
                    },
                    "tools": {
                        "protocolVersion": "2025-11-25",
                        "inlineContent": json.dumps({"tools": rec["tools"]})
                    }
                }
            },
        )
        # Response only has recordArn — extract ID from it
        record_arn = resp["recordArn"]
        record_id = record_arn.rsplit("/", 1)[-1]
        record_ids[rec["name"]] = record_id
        print(f"Created record: {rec['name']} [{record_id}] — status: {resp.get('status', 'DRAFT')}")
    except agentcore_control.exceptions.ConflictException:
        # Record exists — update its description and find it
        existing = agentcore_control.list_registry_records(registryId=REGISTRY_ID)
        for r in existing.get("registryRecords", existing.get("items", [])):
            if r["name"] == rec["name"] and r["status"] == "APPROVED":
                record_ids[rec["name"]] = r["recordId"]
                agentcore_control.update_registry_record(
                    registryId=REGISTRY_ID,
                    recordId=r["recordId"],
                    description={"optionalValue": rec["description"]},
                )
                print(f"Updated record: {rec['name']} [{r['recordId']}] — description now includes Claude Code install instructions")
                break

print(f"\n✅ {len(record_ids)} records created/updated")
for name, rid in record_ids.items():
    print(f"  {name}: {rid}")

## Cell 4: Governance — Approval Workflow

Records must be **approved** before they appear in search results. This simulates an enterprise governance workflow:

```
Developer publishes record → DRAFT
    ↓
Developer submits for review → PENDING_APPROVAL
    ↓
Team lead / curator approves → APPROVED (now searchable)
```

In production, the submit step triggers an **EventBridge** event that can integrate with ticketing systems or security review workflows. Here we play both roles (publisher + curator) to demo the lifecycle.

In [ ]:
for name, record_id in record_ids.items():
    # Check current status
    record = agentcore_control.get_registry_record(registryId=REGISTRY_ID, recordId=record_id)
    status = record["status"]
    print(f"\n--- {name} ---")
    print(f"  Current status: {status}")

    # Step 1: Submit for approval (DRAFT → PENDING_APPROVAL)
    if status == "DRAFT":
        agentcore_control.submit_registry_record_for_approval(
            registryId=REGISTRY_ID, recordId=record_id
        )
        print(f"  Submitted for approval: DRAFT → PENDING_APPROVAL")
        status = "PENDING_APPROVAL"

    # Step 2: Approve (PENDING_APPROVAL → APPROVED)
    if status == "PENDING_APPROVAL":
        agentcore_control.update_registry_record_status(
            registryId=REGISTRY_ID,
            recordId=record_id,
            status="APPROVED",
            statusReason="Approved for Open Insurance tool catalog demo",
        )
        print(f"  Approved: PENDING_APPROVAL → APPROVED")
        status = "APPROVED"

    if status == "APPROVED":
        print(f"  ✅ Record is APPROVED and searchable")
    else:
        print(f"  ⚠️  Unexpected status: {status}")

# Verify final state
print(f"\n--- Final Record Status ---")
records = agentcore_control.list_registry_records(registryId=REGISTRY_ID)
for r in records.get("registryRecords", records.get("items", [])):
    icon = "✅" if r["status"] == "APPROVED" else "⚠️"
    print(f"  {icon} {r['name']}: {r['status']}")

print(f"\n✅ All records approved — they will appear in search results shortly")

## Cell 5: Search the Registry (PKCE Browser Auth — Okta Token)

Since the Registry uses **CUSTOM_JWT** authorization, the search API requires a JWT token — IAM credentials won't work for data plane operations. This matches the agent experience: authenticate with your corporate identity, then search for tools.

We use **Authorization Code + PKCE** — the same flow Claude Code uses:
1. Start a temporary local HTTP server on `localhost:CALLBACK_PORT`
2. Open a browser tab for Okta login
3. Okta redirects back to localhost with an authorization code
4. Exchange the code for a JWT access token

This is more secure than ROPC (no passwords in code) and works with Okta's SPA (public) client.

> **Note:** Newly approved records have an eventual consistency delay (seconds to minutes). We retry with backoff if results are empty.

In [ ]:
import jwt as pyjwt
import hashlib
import base64
import secrets
import webbrowser
import urllib.parse
from http.server import HTTPServer, BaseHTTPRequestHandler
import threading


def pkce_login(client_id, issuer, port, scope="openid profile groups", timeout=120):
    """Run Authorization Code + PKCE flow via browser popup. Returns the access token."""

    # Generate PKCE code verifier + challenge
    code_verifier = secrets.token_urlsafe(64)
    code_challenge = base64.urlsafe_b64encode(
        hashlib.sha256(code_verifier.encode()).digest()
    ).rstrip(b"=").decode()
    state = secrets.token_urlsafe(32)

    redirect_uri = f"http://localhost:{port}/callback"
    result = {"code": None, "error": None}

    class CallbackHandler(BaseHTTPRequestHandler):
        def do_GET(self):
            parsed = urllib.parse.urlparse(self.path)
            params = urllib.parse.parse_qs(parsed.query)

            if "code" in params and params.get("state", [None])[0] == state:
                result["code"] = params["code"][0]
                self.send_response(200)
                self.send_header("Content-Type", "text/html")
                self.end_headers()
                self.wfile.write(b"<html><body><h2>Authentication successful!</h2><p>You can close this tab.</p></body></html>")
            else:
                result["error"] = params.get("error_description", params.get("error", ["Unknown error"]))[0]
                self.send_response(400)
                self.send_header("Content-Type", "text/html")
                self.end_headers()
                self.wfile.write(f"<html><body><h2>Authentication failed</h2><p>{result['error']}</p></body></html>".encode())

        def log_message(self, format, *args):
            pass  # Suppress request logs

    # Start local server
    server = HTTPServer(("localhost", port), CallbackHandler)
    server.timeout = timeout

    # Build authorization URL
    auth_params = urllib.parse.urlencode({
        "client_id": client_id,
        "response_type": "code",
        "scope": scope,
        "redirect_uri": redirect_uri,
        "state": state,
        "code_challenge": code_challenge,
        "code_challenge_method": "S256",
    })
    auth_url = f"{issuer}/v1/authorize?{auth_params}"

    print(f"  Opening browser for Okta login...")
    print(f"  Redirect URI: {redirect_uri}")
    webbrowser.open(auth_url)

    # Wait for callback
    print(f"  Waiting for authentication (timeout: {timeout}s)...")
    server.handle_request()
    server.server_close()

    if result["error"]:
        raise RuntimeError(f"OAuth error: {result['error']}")
    if not result["code"]:
        raise RuntimeError("No authorization code received — did the browser flow complete?")

    # Exchange code for token
    print(f"  Exchanging authorization code for token...")
    token_resp = requests.post(
        f"{issuer}/v1/token",
        data={
            "grant_type": "authorization_code",
            "client_id": client_id,
            "code": result["code"],
            "redirect_uri": redirect_uri,
            "code_verifier": code_verifier,
        },
        headers={"Content-Type": "application/x-www-form-urlencoded"},
    )

    if token_resp.status_code != 200:
        raise RuntimeError(f"Token exchange failed: {token_resp.status_code} — {token_resp.text[:300]}")

    return token_resp.json()["access_token"]


def display_results(results, query):
    """Display search results with tool schemas."""
    print(f"\n{'='*60}")
    print(f"Search: \"{query}\"  →  {len(results)} result(s)")
    print(f"{'='*60}")
    for i, r in enumerate(results, 1):
        print(f"\n  [{i}] {r['name']}")
        print(f"      Type: {r['descriptorType']}  |  Status: {r['status']}  |  Version: {r.get('version', 'N/A')}")
        print(f"      Description: {r.get('description', 'N/A')[:100]}")
        # Parse and show embedded tool schemas
        if r.get("descriptors", {}).get("mcp", {}).get("tools"):
            tools_content = r["descriptors"]["mcp"]["tools"].get("inlineContent", "{}")
            tools = json.loads(tools_content).get("tools", [])
            for t in tools:
                print(f"      Tool: {t['name']} — {t['description'][:80]}...")
                props = t.get("inputSchema", {}).get("properties", {})
                if props:
                    print(f"            Inputs: {', '.join(props.keys())}")


# --- Step 1: Ensure the SPA app has the PKCE redirect URI registered ---
PKCE_REDIRECT_URI = f"http://localhost:{CALLBACK_PORT}/callback"

if OKTA_API_TOKEN:
    print("Checking SPA app redirect URIs...")
    okta_headers = {"Authorization": f"SSWS {OKTA_API_TOKEN}", "Content-Type": "application/json"}
    apps_resp = requests.get(
        f"https://{OKTA_DOMAIN}/api/v1/apps?q=AgentCore&limit=20",
        headers=okta_headers,
    )
    for app in apps_resp.json():
        creds = app.get("credentials", {}).get("oauthClient", {})
        if creds.get("client_id") == SPA_CLIENT_ID:
            current_uris = app["settings"]["oauthClient"].get("redirect_uris", [])
            if PKCE_REDIRECT_URI not in current_uris:
                new_uris = current_uris + [PKCE_REDIRECT_URI]
                app["settings"]["oauthClient"]["redirect_uris"] = new_uris
                resp = requests.put(
                    f"https://{OKTA_DOMAIN}/api/v1/apps/{app['id']}",
                    headers=okta_headers,
                    json=app,
                )
                if resp.status_code == 200:
                    print(f"  Added redirect URI: {PKCE_REDIRECT_URI}")
                else:
                    print(f"  Warning: Could not add redirect URI ({resp.status_code}): {resp.text[:200]}")
            else:
                print(f"  Redirect URI already registered")
            break
else:
    print(f"  No OKTA_API_TOKEN — ensure {PKCE_REDIRECT_URI} is in SPA app redirect URIs")

# --- Step 2: Authenticate via PKCE browser flow ---
print(f"\nAuthenticating via PKCE browser flow...")
access_token = pkce_login(SPA_CLIENT_ID, OKTA_ISSUER, CALLBACK_PORT)

claims = pyjwt.decode(access_token, options={"verify_signature": False})
print(f"\n  Authenticated as: {claims.get('sub', 'unknown')}")
print(f"  Client ID:        {claims.get('cid', claims.get('client_id', 'N/A'))}")
print(f"  Groups:           {claims.get('groups', [])}")

# --- Step 3: Search registry with Bearer token via HTTP ---
REGISTRY_SEARCH_URL = f"https://bedrock-agentcore.{AWS_REGION}.amazonaws.com/registry-records/search"

def search_registry_jwt(query, max_results=10):
    """Search registry with JWT Bearer token, with retry for eventual consistency."""
    for attempt in range(5):
        resp = requests.post(
            REGISTRY_SEARCH_URL,
            headers={
                "Authorization": f"Bearer {access_token}",
                "Content-Type": "application/json",
            },
            json={
                "searchQuery": query,
                "registryIds": [REGISTRY_ARN],
                "maxResults": max_results,
            },
        )
        if resp.status_code != 200:
            print(f"  Search error: {resp.status_code} — {resp.text[:200]}")
            return []
        results = resp.json().get("registryRecords", [])
        if results:
            return results
        wait = 5 * (attempt + 1)
        print(f"  No results yet (eventual consistency) — retrying in {wait}s...")
        time.sleep(wait)
    return []


print(f"\nSearching registry via JWT Bearer token...")
print(f"  Endpoint: {REGISTRY_SEARCH_URL}")

# --- Search 1: Broad semantic search ---
results_all = search_registry_jwt("insurance")
display_results(results_all, "insurance")

# --- Search 2: Targeted keyword search ---
results_policy = search_registry_jwt("policy lookup")
display_results(results_policy, "policy lookup")

# --- Search 3: Semantic search for restricted data ---
results_claims = search_registry_jwt("confidential claims adjuster")
display_results(results_claims, "confidential claims adjuster")

print(f"\n✅ PKCE auth + JWT search working — {len(results_all)} total records found")
print(f"   No client secret needed — public SPA client with browser-based PKCE")

## Cell 6: Configure Claude Code for Registry-First Discovery

This configures Claude Code with **only the Registry** as an MCP server — the Gateway is intentionally left out. When Claude Code searches the Registry and finds tools hosted on the Gateway, it can dynamically add the Gateway connection and authenticate on demand.

This "Registry-first" pattern is valuable for **regulated industries**:
- Users only authenticate to backends they actually need (**least-privilege**)
- Different Gateways can require different Okta scopes or authorization servers
- Audit trails show explicit per-system consent, not blanket access
- The Registry becomes the **single front door** that guides agents to the right tools

### Demo Flow

```
1. Start Claude Code → authenticates to Registry (Okta popup)
2. Ask: "What insurance tools are available?"
   → Registry search returns tools + Gateway URL + Claude Code install instructions
3. Ask: "I need to use the policy lookup tool. Can you add the AgentCore Gateway?"
   → Claude Code reads the install instructions from the record description
   → Runs: claude mcp add-json agentcore-gateway '{"type":"http",...}'
4. Run /reload-plugins → connects to Gateway (Okta popup) — no restart needed!
5. Ask: "Look up policy POL-10042"
   → Invokes tool on Gateway with Cedar enforcement
```

> **Note:** If `03_claude_code_oauth_demo.ipynb` already added the Gateway, this cell removes it to demonstrate the Registry-first flow. The Gateway config is saved and can be re-added later.

In [ ]:
import subprocess

# --- Registry details: use in-memory vars from Cell 2, fall back to config for kernel restart ---
REGISTRY_ID = globals().get("REGISTRY_ID") or config["registry_id"]
REGISTRY_ARN = globals().get("REGISTRY_ARN") or config["registry_arn"]
REGISTRY_MCP_URL = config.get("registry_mcp_url", f"https://bedrock-agentcore.{AWS_REGION}.amazonaws.com/registry/{REGISTRY_ID}/mcp")
OKTA_DISCOVERY_URL = f"{OKTA_ISSUER}/.well-known/openid-configuration"

print(f"Registry MCP endpoint: {REGISTRY_MCP_URL}")

if not SPA_CLIENT_ID:
    print(f"\n⚠️  No SPA client ID found in gateway_config.json")
    print(f"   Run 03_claude_code_oauth_demo.ipynb first to create the Okta SPA app,")
    print(f"   then re-run this notebook from Cell 1.")
else:
    # --- Step 1: Ensure SPA client is in Registry's allowedClients ---
    reg = agentcore_control.get_registry(registryId=REGISTRY_ID)
    current_clients = reg.get("authorizerConfiguration", {}).get("customJWTAuthorizer", {}).get("allowedClients", [])

    if SPA_CLIENT_ID not in current_clients:
        print(f"\nAdding SPA client to Registry's allowedClients...")
        new_clients = current_clients + [SPA_CLIENT_ID]
        agentcore_control.update_registry(
            registryId=REGISTRY_ID,
            authorizerConfiguration={
                "customJWTAuthorizer": {
                    "discoveryUrl": OKTA_DISCOVERY_URL,
                    "allowedAudience": ["api://default"],
                    "allowedClients": new_clients,
                }
            },
        )
        print(f"  Updated allowedClients: {new_clients}")

        for attempt in range(12):
            reg = agentcore_control.get_registry(registryId=REGISTRY_ID)
            if reg["status"] == "READY":
                print(f"  Registry READY")
                break
            time.sleep(5)
            print(f"  Status: {reg['status']} ({(attempt + 1) * 5}s)")
    else:
        print(f"SPA client already in allowedClients")

    # --- Step 2: Remove Gateway from Claude Code (Registry-first demo) ---
    print(f"\nRemoving agentcore-gateway from Claude Code (Registry-first flow)...")
    subprocess.run(["claude", "mcp", "remove", "agentcore-gateway"], capture_output=True)

    # --- Step 3: Add Registry MCP server ---
    MCP_SERVER_NAME = "open-insurance-registry"

    registry_mcp_config = {
        "type": "http",
        "url": REGISTRY_MCP_URL,
        "oauth": {
            "clientId": SPA_CLIENT_ID,
            "callbackPort": CALLBACK_PORT,
            "scope": "openid groups",
            "authorizationUrl": f"{OKTA_ISSUER}/v1/authorize",
            "tokenUrl": f"{OKTA_ISSUER}/v1/token",
        },
    }

    subprocess.run(["claude", "mcp", "remove", MCP_SERVER_NAME], capture_output=True)
    result = subprocess.run(
        ["claude", "mcp", "add-json", MCP_SERVER_NAME, json.dumps(registry_mcp_config)],
        capture_output=True, text=True,
    )

    if result.returncode == 0:
        print(f"✅ Added '{MCP_SERVER_NAME}' MCP server to Claude Code")
    else:
        print(f"⚠️  claude mcp add-json failed: {result.stderr}")

    # --- Step 4: Patch .claude.json for scope/URLs (same workaround as notebook 03) ---
    claude_json_path = os.path.expanduser("~/.claude.json")
    try:
        with open(claude_json_path) as f:
            claude_config = json.load(f)

        project_key = os.getcwd()
        if "projects" in claude_config and project_key in claude_config["projects"]:
            servers = claude_config["projects"][project_key].get("mcpServers", {})
            if MCP_SERVER_NAME in servers:
                servers[MCP_SERVER_NAME]["oauth"]["scope"] = "openid groups"
                servers[MCP_SERVER_NAME]["oauth"]["authorizationUrl"] = f"{OKTA_ISSUER}/v1/authorize"
                servers[MCP_SERVER_NAME]["oauth"]["tokenUrl"] = f"{OKTA_ISSUER}/v1/token"
                with open(claude_json_path, "w") as f:
                    json.dump(claude_config, f, indent=2)
                print("Patched .claude.json with scope and OAuth URLs")
    except FileNotFoundError:
        print("Note: ~/.claude.json not found — Claude Code config may be stored elsewhere")

    # --- Step 5: Save Gateway config for re-adding later ---
    gateway_mcp_config = {
        "type": "http",
        "url": GATEWAY_URL,
        "oauth": {
            "clientId": SPA_CLIENT_ID,
            "callbackPort": CALLBACK_PORT,
            "scope": "openid groups",
            "authorizationUrl": f"{OKTA_ISSUER}/v1/authorize",
            "tokenUrl": f"{OKTA_ISSUER}/v1/token",
        },
    }

    print(f"\n--- Claude Code MCP Servers ---")
    print(f"  ✅ open-insurance-registry:  {REGISTRY_MCP_URL}")
    print(f"  ❌ agentcore-gateway:        REMOVED (Registry-first demo)")
    print(f"\n--- Gateway config (for re-adding during demo) ---")
    print(f"  claude mcp add-json agentcore-gateway '{json.dumps(gateway_mcp_config)}'")
    print(f"\n  After adding, run /mcp or /reload-plugins in Claude Code to connect and authenticate.")
    print(f"  No restart needed — /reload-plugins hot-loads new MCP servers.")

## Test It: Registry-First Discovery Flow

Clear OAuth tokens and start a fresh Claude Code session:

```bash
rm -rf ~/.claude/oauth-tokens/
cd /path/to/demo-centralized-mcp-server
claude
```

### Step 1: Authenticate to Registry only

On startup, Claude Code connects to `open-insurance-registry` — **Okta browser popup** for authentication. The Gateway is **not connected** — only the Registry.

Run `/mcp` to verify:
- `open-insurance-registry` — connected
- `agentcore-gateway` — **not listed**

### Step 2: Discover tools via Registry

```
> What insurance tools are available? Search the registry.
```

Claude Code uses `search_registry_records` and gets back:
- **open-insurance/policy-lookup** — with Claude Code install instructions and Gateway URL
- **open-insurance/claims-data** — with Claude Code install instructions, restricted to claims adjusters

At this point, Claude Code knows **what tools exist**, **where they're hosted**, and **exactly how to install the Gateway** — but it **can't invoke them** yet (no Gateway connection).

### Step 3: Add the Gateway connection (no restart needed!)

Claude Code reads the installation instructions from the Registry record description. Ask it to add the connection:

```
> I need to use the policy lookup tool. Can you add the AgentCore Gateway as an MCP server?
```

Claude Code runs `claude mcp add-json agentcore-gateway '...'` using the config from the Registry metadata.

### Step 4: Connect to Gateway (same session)

Run `/reload-plugins` in Claude Code to hot-load the new MCP server — **no restart needed!**

An Okta popup appears for Gateway authentication. Run `/mcp` to verify both are connected:
- `open-insurance-registry` — connected
- `agentcore-gateway` — connected

### Step 5: Invoke tools via Gateway

```
> Look up insurance policy POL-10042
> Show me the claims summary for claims under 90000
```

Now Claude Code invokes `PolicyLookup___lookup_policy` and `ClaimsData___query_claims` on the Gateway, with Cedar policies enforcing access control.

### The Full Picture

```
     ┌─────────────────────────────────────┐
     │           Claude Code                │
     └──────┬─────────────────────┬─────────┘
            │                     │
   Step 2-3 │ (discover + add)   │ Step 4-5 (reload + invoke)
            │                     │
            ▼                     ▼
     ┌────────────┐       ┌─────────────┐
     │  Registry   │       │   Gateway    │
     │  (catalog)  │       │  (runtime)   │
     │             │       │              │
     │ search_     │       │ PolicyLookup │
     │ registry_   │  ───▶ │ ClaimsData   │
     │ records     │ found │              │
     └─────┬──────┘  here  └──────┬───────┘
           │                      │
     Okta popup #1          Okta popup #2
     (Registry auth)        (Gateway auth)
```

**Key improvement:** The entire discover → install → connect → use flow now happens in a **single Claude Code session** thanks to `/reload-plugins`. No restart required!

**Why this matters for regulated industries:**
- **Least-privilege access** — users only authenticate to systems they actually need
- **Audit trail** — each Okta popup = explicit consent to access that specific system
- **Self-service discovery** — agents find tools AND their install instructions from the Registry
- **Zero manual config** — the Registry record contains everything needed to connect
- **Different auth contexts** — different Gateways can require different Okta scopes, MFA levels, or authorization servers
- **Centralized discovery** — one Registry serves as the front door to all tools across the organization

### Re-adding the Gateway (after demo)

To restore the full setup from `03_claude_code_oauth_demo.ipynb`:

```bash
# The notebook printed the command in Cell 6 output — copy/paste it, or run:
claude mcp add-json agentcore-gateway '{"type":"http","url":"<gateway-url>","oauth":{...}}'
```

## Registry Skills: Discover, Enable, and Use Agent Workflows

So far we've published **MCP records** — tool metadata that tells agents _what to call_. The Registry also supports a second descriptor type: **AGENT_SKILLS** — reusable workflows that tell agents _how to reason_.

| Descriptor Type | What it stores | Purpose |
|----------------|---------------|---------|
| **MCP** | Tool name, input schema, server info | "Here's a tool you can call" |
| **AGENT_SKILLS** | Markdown instructions + JSON definition | "Here's a workflow to follow" |

```
Registry
├── MCP Records (tools — what to call)
│   ├── open-insurance-policy-lookup     ← tool schema + Gateway URL
│   └── open-insurance-claims-data       ← tool schema + Gateway URL
│
└── AGENT_SKILLS Records (workflows — how to reason)
    └── open-insurance-claim-triage      ← markdown instructions + metadata
        "Given a claim ID, gather data from
         PolicyLookup + ClaimsData, assess
         severity, flag risks, produce report"
```

This enables a **"skill store" pattern**: agents search the Registry, find a useful workflow, install it locally, and start using it — all in one session.

### How this maps to Claude Code

Claude Code supports **custom slash commands** — markdown files in `.claude/commands/` that act as reusable prompt templates. When an agent discovers an `AGENT_SKILLS` record in the Registry, it can:

1. Extract the `skillMd` content (markdown instructions)
2. Write it to `.claude/commands/<skill-name>.md`
3. Invoke it immediately as `/<skill-name>` — no restart needed

This is the **Discover → Enable → Use** flow we'll demo next.

### Publish a Skill Record — Claim Triage Workflow

We'll publish a **claim-triage** skill that orchestrates the existing PolicyLookup and ClaimsData tools into a structured triage workflow.

**Important distinction:** This skill is a **standalone reasoning workflow**, not part of the AgentCore Gateway MCP server. It is published as an `AGENT_SKILLS` record — pure markdown instructions that tell an agent _how to reason_. The skill _depends on_ the Gateway-hosted MCP tools (PolicyLookup, ClaimsData), but it is not hosted on or served by the Gateway itself.

| Layer | What it is | Where it lives |
|-------|-----------|----------------|
| **Skill** (AGENT_SKILLS) | Markdown reasoning workflow | Registry only — installed locally as a slash command |
| **Tools** (MCP) | PolicyLookup, ClaimsData | Gateway — invoked at runtime via MCP |

The skill definition has two parts:
- **`skillMd`** — The markdown instructions (what Claude Code installs as a slash command)
- **`skillDefinition`** — JSON metadata (name, version, parameters, required tools)

In [ ]:
# --- Skill markdown (what gets installed as a Claude Code slash command) ---
# Note: Registry requires skillMd to start with YAML frontmatter delimited by '---'

CLAIM_TRIAGE_SKILL_MD = """\
---
name: claim-triage
description: Triage an insurance claim by gathering data, assessing severity, and producing a structured report.
version: 1.0.0
---

Triage an insurance claim by gathering data from available tools, assessing severity, and producing a structured triage report.

Arguments: $ARGUMENTS

## Triage Process

### Step 1 — Identify the Claim
Parse the input to extract a claim ID (e.g., CLM-2025-0067) or policyholder name.
If the input is ambiguous, ask for clarification before proceeding.

### Step 2 — Gather Claim Details
Use the `ClaimsData___query_claims` tool to retrieve the full claim record.
Note the claim type, status, amount claimed, and any adjuster notes.

### Step 3 — Retrieve Policy Context
Extract the policy number from the claim record and use `PolicyLookup___lookup_policy`
to retrieve the associated policy. Note coverage limits, policy status, and expiry date.

### Step 4 — Assess Severity
Classify the claim into one of these severity tiers based on the data gathered:

| Tier | Criteria | SLA |
|------|----------|-----|
| **P1 — Critical** | Amount > $150K, OR status contains "LEGAL HOLD", OR policy expired/under review | 4 hours |
| **P2 — High** | Amount > $50K, OR status "Under Investigation", OR multiple claims on same policy | 24 hours |
| **P3 — Standard** | Amount $10K–$50K with no complicating factors | 3 business days |
| **P4 — Low** | Amount < $10K, straightforward claim with clear liability | 5 business days |

### Step 5 — Check for Red Flags
Flag any of the following if present:
- Claim amount exceeds policy coverage limits
- Multiple open claims from the same policyholder
- Claim filed within 90 days of policy inception
- "Under Investigation" or "LEGAL HOLD" in adjuster notes
- Policy status is not "Active"

### Step 6 — Produce Triage Report
Output a structured report in this exact format:

```
=== CLAIM TRIAGE REPORT ===
Claim ID:       [claim_id]
Policyholder:   [name]
Policy:         [policy_number] ([type], [status])
Claim Type:     [type]
Amount Claimed: [amount]
Coverage Limit: [relevant coverage from policy]

SEVERITY:       [P1/P2/P3/P4] — [tier name]
SLA:            [response time]

RED FLAGS:
- [list any flags, or "None identified"]

RECOMMENDED ACTIONS:
1. [First action based on severity and flags]
2. [Second action]
3. [Third action if applicable]

NOTES:
[Any additional context from adjuster notes or policy review]
===========================
```

If the AgentCore Gateway tools are not available, inform the user that this skill
requires the PolicyLookup and ClaimsData tools, and suggest connecting to the Gateway first.
"""

# --- Skill definition (JSON metadata) ---

CLAIM_TRIAGE_SKILL_DEF = {
    "name": "claim-triage",
    "description": "Triage an insurance claim by gathering data, assessing severity, and producing a structured report with recommended actions.",
    "version": "1.0.0",
    "parameters": {
        "type": "object",
        "properties": {
            "input": {
                "type": "string",
                "description": "Claim ID (e.g., CLM-2025-0067), policyholder name, or description of the claim to triage"
            }
        },
        "required": ["input"]
    },
    "requiredTools": [
        "ClaimsData___query_claims",
        "PolicyLookup___lookup_policy"
    ]
}

# --- Build skill description with Claude Code installation instructions ---

SKILL_DESCRIPTION = (
    "Claim triage reasoning workflow for the Open Insurance platform. "
    "Guides agents through a structured triage process: gather claim and policy data, "
    "assess severity (P1–P4), check for red flags, and produce a formatted triage report.\n\n"
    "## Claude Code Installation\n\n"
    "This is an AGENT_SKILLS record (a reasoning workflow, not an MCP server). "
    "To install it as a Claude Code slash command:\n\n"
    "1. Search the registry and extract the `skillMd` content from this record\n"
    "2. Write it to `.claude/commands/claim-triage.md` in your project directory\n"
    "3. The `/claim-triage` command is immediately available (no restart needed)\n\n"
    "This skill depends on the PolicyLookup and ClaimsData MCP tools hosted on the AgentCore Gateway. "
    "If the Gateway is not yet connected, add it first:\n\n"
    "```\n"
    f"claude mcp add-json agentcore-gateway '{GATEWAY_MCP_CONFIG_JSON}'\n"
    "```\n\n"
    "Then run `/mcp` or `/reload-plugins` in Claude Code to connect without restarting the session.\n\n"
    "Required tools: `PolicyLookup___lookup_policy`, `ClaimsData___query_claims`\n"
    f"Gateway URL: {GATEWAY_URL}\n"
    "Auth: Okta JWT (CUSTOM_JWT authorizer), Cedar ENFORCE mode"
)

# --- Publish to Registry ---

print("Publishing claim-triage skill to Registry...")
print(f"  Descriptor type: AGENT_SKILLS")
print(f"  Skill markdown:  {len(CLAIM_TRIAGE_SKILL_MD)} chars")
print(f"  Required tools:  {CLAIM_TRIAGE_SKILL_DEF['requiredTools']}")

try:
    resp = agentcore_control.create_registry_record(
        registryId=REGISTRY_ID,
        name="open-insurance-claim-triage",
        description=SKILL_DESCRIPTION,
        descriptorType="AGENT_SKILLS",
        recordVersion="1.0.0",
        descriptors={
            "agentSkills": {
                "skillMd": {
                    "inlineContent": CLAIM_TRIAGE_SKILL_MD
                },
                "skillDefinition": {
                    "schemaVersion": "0.1.0",
                    "inlineContent": json.dumps(CLAIM_TRIAGE_SKILL_DEF)
                }
            }
        },
    )
    skill_record_arn = resp["recordArn"]
    skill_record_id = skill_record_arn.rsplit("/", 1)[-1]
    print(f"\n✅ Created skill record: open-insurance-claim-triage [{skill_record_id}]")
    print(f"   Status: DRAFT (needs approval before it's searchable)")
except agentcore_control.exceptions.ConflictException:
    # Record exists — update its description and find it
    print("Skill record already exists — updating description...")
    existing = agentcore_control.list_registry_records(registryId=REGISTRY_ID)
    for r in existing.get("registryRecords", existing.get("items", [])):
        if r["name"] == "open-insurance-claim-triage" and r["status"] == "APPROVED":
            skill_record_id = r["recordId"]
            agentcore_control.update_registry_record(
                registryId=REGISTRY_ID,
                recordId=skill_record_id,
                description={"optionalValue": SKILL_DESCRIPTION},
            )
            print(f"  Updated: open-insurance-claim-triage [{skill_record_id}] — description now includes Claude Code install instructions")
            break
    else:
        # Fallback: find any matching record
        for r in existing.get("registryRecords", existing.get("items", [])):
            if r["name"] == "open-insurance-claim-triage":
                skill_record_id = r["recordId"]
                print(f"  Found existing: open-insurance-claim-triage [{skill_record_id}] — status: {r['status']}")
                break
        else:
            raise RuntimeError("ConflictException but could not find skill record")

### Approve the Skill Record

Skills go through the same governance workflow as MCP records: **DRAFT → PENDING_APPROVAL → APPROVED**. This is deliberate — an organization wants to review skills before they become discoverable, because skills encode business logic and reasoning workflows, not just tool schemas.

In [ ]:
# --- Approve the skill record ---

record = agentcore_control.get_registry_record(registryId=REGISTRY_ID, recordId=skill_record_id)
status = record["status"]
print(f"--- open-insurance-claim-triage ---")
print(f"  Current status: {status}")

if status == "DRAFT":
    agentcore_control.submit_registry_record_for_approval(
        registryId=REGISTRY_ID, recordId=skill_record_id
    )
    print(f"  Submitted for approval: DRAFT → PENDING_APPROVAL")
    status = "PENDING_APPROVAL"

if status == "PENDING_APPROVAL":
    agentcore_control.update_registry_record_status(
        registryId=REGISTRY_ID,
        recordId=skill_record_id,
        status="APPROVED",
        statusReason="Approved claim-triage skill for Open Insurance demo",
    )
    print(f"  Approved: PENDING_APPROVAL → APPROVED")
    status = "APPROVED"

if status == "APPROVED":
    print(f"  ✅ Skill record is APPROVED and searchable")

# --- Show all records (MCP + Skills) ---
print(f"\n--- All Registry Records ---")
records = agentcore_control.list_registry_records(registryId=REGISTRY_ID)
for r in records.get("registryRecords", records.get("items", [])):
    if r["status"] == "APPROVED":
        icon = "🔧" if r["descriptorType"] == "MCP" else "📋"
        print(f"  {icon} {r['name']}: {r['descriptorType']} — {r['status']}")

print(f"\n✅ Skill approved — Registry now contains both MCP tools and AGENT_SKILLS workflows")

### Search for Skills

The same search API returns both MCP and AGENT_SKILLS records. An agent searching for "claim triage" will find the skill alongside the MCP tools it depends on — giving it everything it needs to both _understand the workflow_ and _discover the tools_.

In [ ]:
# --- Refresh JWT token via PKCE (may have expired since Cell 5) ---
print("Refreshing JWT token via PKCE browser flow...")
try:
    access_token = pkce_login(SPA_CLIENT_ID, OKTA_ISSUER, CALLBACK_PORT)
    claims = pyjwt.decode(access_token, options={"verify_signature": False})
    print(f"  Authenticated as: {claims.get('sub', 'unknown')}")
except Exception as e:
    print(f"  Token refresh failed: {e} — using existing token")


def display_results_v2(results, query):
    """Display search results — handles both MCP and AGENT_SKILLS records."""
    print(f"\n{'='*60}")
    print(f"Search: \"{query}\"  →  {len(results)} result(s)")
    print(f"{'='*60}")
    for i, r in enumerate(results, 1):
        dtype = r.get("descriptorType", "?")
        icon = "🔧" if dtype == "MCP" else "📋" if dtype == "AGENT_SKILLS" else "📦"
        print(f"\n  {icon} [{i}] {r['name']}")
        print(f"      Type: {dtype}  |  Status: {r['status']}  |  Version: {r.get('version', 'N/A')}")
        print(f"      Description: {r.get('description', 'N/A')[:100]}")

        # MCP records — show tool schemas
        if r.get("descriptors", {}).get("mcp", {}).get("tools"):
            tools_content = r["descriptors"]["mcp"]["tools"].get("inlineContent", "{}")
            tools = json.loads(tools_content).get("tools", [])
            for t in tools:
                print(f"      Tool: {t['name']} — {t['description'][:80]}...")
                props = t.get("inputSchema", {}).get("properties", {})
                if props:
                    print(f"            Inputs: {', '.join(props.keys())}")

        # AGENT_SKILLS records — show skill content preview and metadata
        if r.get("descriptors", {}).get("agentSkills"):
            skill_md = r["descriptors"]["agentSkills"].get("skillMd", {}).get("inlineContent", "")
            skill_def_raw = r["descriptors"]["agentSkills"].get("skillDefinition", {}).get("inlineContent", "{}")
            if skill_md:
                # Show first meaningful line after frontmatter
                lines = skill_md.strip().split("\n")
                preview = next((l for l in lines if l.strip() and not l.startswith("---") and not l.startswith("name:") and not l.startswith("description:") and not l.startswith("version:")), lines[0])
                print(f"      Skill: {preview[:90]}...")
                print(f"      Size:  {len(skill_md)} chars")
            try:
                skill_def = json.loads(skill_def_raw)
                if skill_def.get("requiredTools"):
                    print(f"      Required tools: {skill_def['requiredTools']}")
                if skill_def.get("parameters", {}).get("properties"):
                    print(f"      Parameters: {list(skill_def['parameters']['properties'].keys())}")
            except json.JSONDecodeError:
                pass


# --- Search 1: "claim triage" — should find the skill + related MCP records ---
print("Searching for 'claim triage'...")
results = search_registry_jwt("claim triage")
display_results_v2(results, "claim triage")

# --- Search 2: Broad search — shows MCP + AGENT_SKILLS side by side ---
print("\n\nSearching for 'insurance' (all record types)...")
results_all = search_registry_jwt("insurance")
display_results_v2(results_all, "insurance")

# --- Summary ---
mcp_count = sum(1 for r in results_all if r.get("descriptorType") == "MCP")
skill_count = sum(1 for r in results_all if r.get("descriptorType") == "AGENT_SKILLS")
print(f"\n✅ Registry search returns both types: {mcp_count} MCP tools + {skill_count} AGENT_SKILLS workflow(s)")

### Demo — Discover, Enable, and Use a Skill in Claude Code

This is the key demo: Claude Code discovers a skill from the Registry, installs it locally, and uses it — **all in one session, no restart needed**.

```
Claude Code Session
│
├── 1. DISCOVER: Search Registry for skills
│   └── search_registry_records("claim triage")
│       → Returns skill record with skillMd content + install instructions
│
├── 2. ENABLE: Claude Code extracts skillMd and installs locally
│   └── Writes skillMd to .claude/commands/claim-triage.md
│       → Skill immediately available as /claim-triage
│       → Claude Code reads commands/ at invocation time, not startup
│
├── 3. CONNECT: Add Gateway if not already connected
│   └── claude mcp add-json agentcore-gateway '...' (from record description)
│   └── /reload-plugins → hot-loads the new MCP server (no restart!)
│
└── 4. USE: Invoke the skill
    └── /claim-triage CLM-2025-0067
        → Claude follows the triage workflow
        → Calls PolicyLookup and ClaimsData via Gateway
        → Produces structured triage report
```

### Prerequisites
- The Registry-first Claude Code configuration cell has been run (Registry MCP server configured — Registry-only mode)
- The skill publish and approve cells above have been run (skill record APPROVED)
- AgentCore Gateway can be added mid-session via `/reload-plugins`

### Step 1: Start Claude Code
```bash
cd /path/to/demo-centralized-mcp-server
claude
```
On startup, Claude Code connects to `open-insurance-registry` (Okta popup for auth).

### Step 2: Discover the skill
```
> Search the registry for skills related to insurance claim triage
```

Claude Code uses `search_registry_records` and finds `open-insurance-claim-triage` with type `AGENT_SKILLS`. The record description includes installation instructions for both the skill and the Gateway.

### Step 3: Enable the skill (same session — no restart)
```
> Install that claim-triage skill so I can use it as a slash command
```

Claude Code will:
1. Extract the `skillMd` content from the Registry record
2. Write it to `.claude/commands/claim-triage.md`
3. Confirm the skill is now available as `/claim-triage`

**Key point:** Claude Code reads `.claude/commands/` at invocation time, not at startup. The skill is available immediately.

### Step 4: Connect the Gateway (if not already connected)
```
> Add the AgentCore Gateway as an MCP server so the skill can call the tools
```

Claude Code reads the install command from the record description and runs it. Then run `/reload-plugins` to hot-load the Gateway — **no restart needed!**

### Step 5: Use the skill
```
> /claim-triage CLM-2025-0067 also compare this with other claims under 99000
```

Claude follows the triage workflow:
1. Queries ClaimsData for CLM-2025-0067 (David Park, $89,500 fire damage, Under Investigation)
2. Looks up POL-10156 (Business Insurance, Under Review)
3. Assesses severity as **P2 — High** (amount > $50K + Under Investigation)
4. Flags: policy Under Review, fire damage under investigation
5. Produces the structured triage report with recommended actions

### The Complete Picture

```
     ┌─────────────────────────────────────────┐
     │              Claude Code                  │
     │                                           │
     │  1. Search Registry → find claim-triage   │
     │  2. Write .claude/commands/claim-triage.md│
     │  3. Add Gateway + /reload-plugins         │
     │  4. /claim-triage CLM-2025-0067           │
     └───────┬───────────────────────┬───────────┘
             │                       │
             ▼                       ▼
      ┌────────────┐         ┌─────────────┐
      │  Registry   │         │   Gateway    │
      │  (discover) │         │  (execute)   │
      │             │         │              │
      │ claim-triage│    ───▶ │ PolicyLookup │
      │ (AGENT_SKILL│  skill  │ ClaimsData   │
      │  record)    │  calls  │              │
      └─────────────┘  these  └──────────────┘
```

**Single-session flow:** Discover → Install skill → Add Gateway → `/reload-plugins` → Use. No restarts at any point.

### Preview Limitation: Skills are Inline Markdown Only — No Code Artifacts

The `AGENT_SKILLS` descriptor stores exactly two inline string fields:

| Field | Type | Content |
|-------|------|---------|
| `skillMd.inlineContent` | string | Markdown instructions (the skill itself) |
| `skillDefinition.inlineContent` | string | JSON metadata (name, version, parameters) |

**There is no file attachment mechanism, no binary upload, no code artifact storage.**

This means a skill like `claim-triage` that is purely a _reasoning workflow_ works perfectly — it's just markdown instructions. But many powerful skills need more:

```
claim-triage/                          What the Registry can store
├── claim-triage.md                    ✅ skillMd.inlineContent
├── skill-definition.json              ✅ skillDefinition.inlineContent
├── triage_scoring.py                  ❌ NOT supported
├── report_template.html               ❌ NOT supported
├── business_rules.yaml                ❌ NOT supported
└── tests/
    └── test_triage.py                 ❌ NOT supported
```

### What works well as markdown-only skills

- **Guided reasoning workflows** — triage, analysis, decision trees (like our claim-triage)
- **Structured prompt templates** — report generation, code review checklists
- **Process documentation** — step-by-step operational runbooks
- **Configuration guides** — environment setup, deployment procedures

### What needs more than markdown

- **Skills with scoring models** — fraud risk calculators, priority algorithms in Python
- **Template-based generators** — PDF reports from Jinja2, Excel from openpyxl
- **Skills with test fixtures** — validation data, expected output snapshots
- **Multi-file tools** — CLI utilities, Lambda functions, Docker compose stacks

### Workaround: External artifact references

Today, you can reference external code artifacts in the skill definition or markdown:

```json
{
  "name": "claim-triage-advanced",
  "codeRepository": "https://github.com/org/insurance-skills/tree/main/claim-triage",
  "artifactBucket": "s3://org-skills/claim-triage/v1.0.0/"
}
```

The skill markdown can include instructions to fetch and install these artifacts. But this is a manual convention, not a Registry-enforced contract.

> **Positioning:** The Agent Registry is in preview. The `AGENT_SKILLS` descriptor type is the first step toward a skill distribution system. A natural evolution would be to support file attachments (S3 references) or artifact bundles — similar to how container registries store images alongside metadata. For now, the Registry excels at _discovering_ skills; _distributing_ their full implementation requires an external artifact store.

## Save Configuration

Saves Registry resource IDs (including skill records from the previous section) to `gateway_config.json` for reference and cleanup.

In [ ]:
# Reload config to avoid overwriting changes from other notebooks
with open("gateway_config.json") as f:
    config = json.load(f)

config["registry_id"] = REGISTRY_ID
config["registry_arn"] = REGISTRY_ARN
config["registry_mcp_url"] = REGISTRY_MCP_URL
config["registry_records"] = record_ids
config["registry_skill_records"] = {"open-insurance-claim-triage": skill_record_id}

with open("gateway_config.json", "w") as f:
    json.dump(config, f, indent=2)

print(f"✅ Config saved to gateway_config.json")
print(f"\n  registry_id:            {REGISTRY_ID}")
print(f"  registry_arn:           {REGISTRY_ARN}")
print(f"  registry_mcp_url:       {REGISTRY_MCP_URL}")
print(f"  registry_records:       {list(record_ids.keys())}")
print(f"  registry_skill_records: open-insurance-claim-triage [{skill_record_id}]")

## Restore Claude Code to Gateway-Only Mode

After running the Registry-first demo, use this cell to **restore** Claude Code to the original Gateway-only setup — removing the Registry MCP server and re-adding the Gateway.

This lets you quickly toggle between demo modes:
- **Configure Claude Code cell above** → Registry-first (Registry only, no Gateway)
- **This cell** → Gateway-only (Gateway only, no Registry)

After running this cell, run `/reload-plugins` in Claude Code to pick up the changes — no restart needed.

In [ ]:
import os
import json
import subprocess

# --- Load config ---
with open("gateway_config.json") as f:
    cfg = json.load(f)

spa_client_id = cfg.get("okta_spa_client_id", "")
if not spa_client_id:
    raise SystemExit("No SPA client ID found — run 03_claude_code_oauth_demo.ipynb first")

# --- Step 1: Remove Registry MCP server ---
result = subprocess.run(["claude", "mcp", "remove", "open-insurance-registry"], capture_output=True, text=True)
if result.returncode == 0:
    print("Removed open-insurance-registry from Claude Code")
else:
    print(f"open-insurance-registry not found (may already be removed)")

# --- Step 2: Re-add Gateway MCP server ---
gateway_mcp_config = {
    "type": "http",
    "url": cfg["gateway_url"],
    "oauth": {
        "clientId": spa_client_id,
        "callbackPort": cfg.get("claude_code_callback_port", 8400),
        "scope": "openid groups",
        "authorizationUrl": f"{cfg['okta_issuer']}/v1/authorize",
        "tokenUrl": f"{cfg['okta_issuer']}/v1/token",
    },
}

subprocess.run(["claude", "mcp", "remove", "agentcore-gateway"], capture_output=True)
result = subprocess.run(
    ["claude", "mcp", "add-json", "agentcore-gateway", json.dumps(gateway_mcp_config)],
    capture_output=True, text=True,
)

if result.returncode == 0:
    print("Added agentcore-gateway to Claude Code")
else:
    print(f"⚠️  Failed to add gateway: {result.stderr}")

# --- Step 3: Patch .claude.json for scope/URLs ---
claude_json_path = os.path.expanduser("~/.claude.json")
try:
    with open(claude_json_path) as f:
        claude_config = json.load(f)

    project_key = os.getcwd()
    if "projects" in claude_config and project_key in claude_config["projects"]:
        servers = claude_config["projects"][project_key].get("mcpServers", {})
        if "agentcore-gateway" in servers:
            servers["agentcore-gateway"]["oauth"]["scope"] = "openid groups"
            servers["agentcore-gateway"]["oauth"]["authorizationUrl"] = f"{cfg['okta_issuer']}/v1/authorize"
            servers["agentcore-gateway"]["oauth"]["tokenUrl"] = f"{cfg['okta_issuer']}/v1/token"
            with open(claude_json_path, "w") as f:
                json.dump(claude_config, f, indent=2)
            print("Patched .claude.json with scope and OAuth URLs")
except FileNotFoundError:
    print("Note: ~/.claude.json not found")

# --- Summary ---
print(f"\n--- Claude Code MCP Servers ---")
print(f"  ✅ agentcore-gateway:        {cfg['gateway_url']}")
print(f"  ❌ open-insurance-registry:  REMOVED")
print(f"\nRun /reload-plugins in Claude Code to connect — no restart needed.")

## Clean Up Registry-Installed Skills

Removes skill `.md` files that were installed from the Registry into `.claude/commands/`, but **preserves the commands directory** so any non-Registry commands you've added manually remain intact.

In [ ]:
import os

# Skills installed from the Registry — add any new skill filenames here
REGISTRY_INSTALLED_SKILLS = [
    "claim-triage.md",
]

commands_dir = os.path.join(os.getcwd(), ".claude", "commands")

if not os.path.isdir(commands_dir):
    print(f"Nothing to clean — {commands_dir} does not exist")
else:
    removed = []
    for skill_file in REGISTRY_INSTALLED_SKILLS:
        skill_path = os.path.join(commands_dir, skill_file)
        if os.path.exists(skill_path):
            os.remove(skill_path)
            removed.append(skill_file)
            print(f"  Removed: {skill_file}")
        else:
            print(f"  Already absent: {skill_file}")

    # Show what's left in the directory
    remaining = os.listdir(commands_dir) if os.path.isdir(commands_dir) else []
    remaining = [f for f in remaining if not f.startswith(".")]  # ignore hidden files

    if removed:
        print(f"\n✅ Removed {len(removed)} registry-installed skill(s)")
    else:
        print(f"\nNo registry-installed skills found to remove")

    if remaining:
        print(f"   Commands dir preserved — {len(remaining)} file(s) remain: {remaining}")
    else:
        print(f"   Commands dir preserved (empty)")

## Cleanup

Removes the Registry, its records (MCP + Skills), installed skill files, and the Claude Code MCP config. Does **not** remove the Gateway or Lambda targets (use `01_setup.ipynb` cleanup for that).

In [ ]:
# ⚠️  CLEANUP — This deletes the Registry and its records. Only run when done!

import json
import os
import boto3
import subprocess

with open("gateway_config.json") as f:
    cfg = json.load(f)

region = cfg["aws_region"]
agentcore = boto3.client("bedrock-agentcore-control", region_name=region)

registry_id = cfg.get("registry_id", "")
registry_records = cfg.get("registry_records", {})
registry_skill_records = cfg.get("registry_skill_records", {})

print("Cleaning up Registry resources...\n")

# 1. Delete MCP registry records
for name, record_id in registry_records.items():
    try:
        agentcore.delete_registry_record(registryId=registry_id, recordId=record_id)
        print(f"  Deleted MCP record: {name} ({record_id})")
    except Exception as e:
        print(f"  Skip MCP record {name}: {e}")

# 2. Delete skill registry records
for name, record_id in registry_skill_records.items():
    try:
        agentcore.delete_registry_record(registryId=registry_id, recordId=record_id)
        print(f"  Deleted skill record: {name} ({record_id})")
    except Exception as e:
        print(f"  Skip skill record {name}: {e}")

# 3. Delete registry
if registry_id:
    try:
        agentcore.delete_registry(registryId=registry_id)
        print(f"  Deleted Registry: {registry_id}")
    except Exception as e:
        print(f"  Skip Registry: {e}")

# 4. Remove installed skill file
skill_path = os.path.join(os.getcwd(), ".claude", "commands", "claim-triage.md")
if os.path.exists(skill_path):
    os.remove(skill_path)
    print(f"  Removed installed skill: {skill_path}")
    # Remove commands dir if empty
    commands_dir = os.path.dirname(skill_path)
    if os.path.isdir(commands_dir) and not os.listdir(commands_dir):
        os.rmdir(commands_dir)
        print(f"  Removed empty commands directory")

# 5. Remove Registry from Claude Code
subprocess.run(["claude", "mcp", "remove", "open-insurance-registry"], capture_output=True)
print("  Removed open-insurance-registry from Claude Code")

# 6. Restore Gateway in Claude Code (removed in Cell 6 for Registry-first demo)
spa_client_id = cfg.get("okta_spa_client_id", "")
if spa_client_id:
    gateway_config_restore = {
        "type": "http",
        "url": cfg["gateway_url"],
        "oauth": {
            "clientId": spa_client_id,
            "callbackPort": cfg.get("claude_code_callback_port", 8400),
            "scope": "openid groups",
            "authorizationUrl": f"{cfg['okta_issuer']}/v1/authorize",
            "tokenUrl": f"{cfg['okta_issuer']}/v1/token",
        },
    }
    subprocess.run(["claude", "mcp", "remove", "agentcore-gateway"], capture_output=True)
    result = subprocess.run(
        ["claude", "mcp", "add-json", "agentcore-gateway", json.dumps(gateway_config_restore)],
        capture_output=True, text=True,
    )
    if result.returncode == 0:
        print("  Restored agentcore-gateway in Claude Code")
    else:
        print(f"  Could not restore gateway: {result.stderr}")

# 7. Clean up gateway_config.json
for key in ["registry_id", "registry_arn", "registry_mcp_url", "registry_records", "registry_skill_records"]:
    cfg.pop(key, None)

with open("gateway_config.json", "w") as f:
    json.dump(cfg, f, indent=2)

print("\n✅ Registry cleanup complete — Gateway restored in Claude Code")